# 2.6 Tool use: multi-turn loops, multiple tools, built-in tools

Step 2, lesson 6 of [Building with the Claude API](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287750) (Anthropic Academy).
Exam domains: **1 Agentic Architecture & Orchestration** (the agentic loop), **2 Tool Design & MCP Integration** (multiple tools, built-in tools, strict schemas) and **4 Prompt Engineering & Structured Output** (tools as a way to get structured data).

What this notebook covers, in the course's order:

1. The reminders tools from 2.5
2. The multi-turn tool loop
3. Multiple tools and parallel tool calls
4. The course's "batch tool", and why it is rarely needed now
5. Tools for structured data (and the forced `tool_choice` that current models reject)
6. Fine-grained tool streaming (one line: out of scope)
7. The text editor tool (Anthropic-defined, runs on your machine)
8. The web search tool (runs on Anthropic's servers)

> **Model used here:** `claude-sonnet-5-5`. **Cost:** about 11 requests, one of them with web search ($10 per 1,000 searches, plus tokens).

In [1]:
import json
import shutil
from datetime import datetime, timedelta
from pathlib import Path

import anthropic
from anthropic import Anthropic
from anthropic.types import Message, ToolParam
from dateutil.relativedelta import relativedelta
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
model = "claude-sonnet-5-5"


def text_of(message):
    """Return the text of the first text block in a response ("" if there is none)."""
    return next((block.text for block in message.content if block.type == "text"), "")


def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, content):
    if isinstance(content, Message):
        content = content.content
    messages.append({"role": "assistant", "content": content})


def chat(messages, system=None, tools=None, tool_choice=None, max_tokens=3000):
    params = {"model": model, "max_tokens": max_tokens, "messages": messages}
    if system:
        params["system"] = system
    if tools:
        params["tools"] = tools
    if tool_choice:
        params["tool_choice"] = tool_choice
    return client.messages.create(**params)

## 1. The reminders tools from 2.5

The same three functions and schemas as in [2.5](05-tool-use-basics.ipynb), condensed.

In [2]:
UNITS = ("seconds", "minutes", "hours", "days", "weeks", "months", "years")
REMINDERS = []


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"):
    if unit not in UNITS:
        raise ValueError(f"unit must be one of {', '.join(UNITS)}; got {unit!r}")
    try:
        start = datetime.strptime(datetime_str, input_format)
    except ValueError:
        raise ValueError(f"datetime_str {datetime_str!r} does not match input_format {input_format!r}")
    if unit in ("months", "years"):
        if duration != int(duration):
            raise ValueError("months and years need a whole number")
        result = start + relativedelta(**{unit: int(duration)})
    else:
        result = start + timedelta(**{unit: duration})
    return result.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    if not content.strip():
        raise ValueError("content cannot be empty")
    try:
        when = datetime.fromisoformat(timestamp)
    except ValueError:
        raise ValueError(f"timestamp must be ISO 8601, e.g. 2050-06-27T09:00:00; got {timestamp!r}")
    REMINDERS.append({"content": content, "timestamp": when.isoformat()})
    return f"Reminder set for {when.isoformat()}: {content}"


reminder_tools: list[ToolParam] = [
    {
        "name": "get_current_datetime",
        "description": (
            "Returns the current local date and time as a string. Use it whenever the user refers to 'now', "
            "'today' or any time relative to the present, because you do not know the current date."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"date_format": {"type": "string", "description": "Python strftime format. Default '%Y-%m-%d %H:%M:%S'."}},
            "required": [],
        },
    },
    {
        "name": "add_duration_to_datetime",
        "description": (
            "Adds a duration to a date or datetime and returns the result, formatted like "
            "'Thursday, June 27, 2050 12:00:00 AM'. Use it for any date arithmetic instead of calculating dates yourself."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "datetime_str": {"type": "string", "description": "The starting date or datetime, matching input_format."},
                "duration": {"type": "number", "description": "How many units to add (negative to subtract)."},
                "unit": {"type": "string", "enum": list(UNITS)},
                "input_format": {"type": "string", "description": "Python strptime format of datetime_str. Default '%Y-%m-%d'."},
            },
            "required": ["datetime_str", "duration", "unit"],
        },
    },
    {
        "name": "set_reminder",
        "description": (
            "Creates a reminder that notifies the user at the given time. Use it only once you know the exact "
            "date and time; work it out with the other tools first. Returns a confirmation."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "content": {"type": "string", "description": "What to remind the user about."},
                "timestamp": {"type": "string", "description": "ISO 8601: 'YYYY-MM-DDTHH:MM:SS'."},
            },
            "required": ["content", "timestamp"],
        },
    },
]

TOOL_FUNCTIONS = {
    "get_current_datetime": get_current_datetime,
    "add_duration_to_datetime": add_duration_to_datetime,
    "set_reminder": set_reminder,
}

## 2. The multi-turn tool loop

Real requests need several tool calls in a row: "remind me in 6 months" means *get today's date*, then *add 6 months*, then *set the reminder*. Claude can only ask for the next step after seeing the result of the previous one, so the code keeps calling the API in a loop:

1. send the conversation (with the tools);
2. append Claude's **full** reply to the history;
3. if `stop_reason` is `"tool_use"`: run **every** tool call in the reply, append **one** user message holding all the `tool_result` blocks, and go back to 1;
4. otherwise (`"end_turn"`) Claude has finished: stop.

Two helpers from the course do the work: `run_tools` turns a reply's `tool_use` blocks into `tool_result` blocks (an exception becomes `is_error: True`, so Claude sees it and can recover), and `run_conversation` is the loop.

> **Beyond the course.** `max_iterations` is a safety net against a loop that never ends. It is *not* how the loop normally stops: that is `stop_reason`.

In [3]:
def run_tools(message, registry):
    """Run every tool_use block in a reply; return the matching tool_result blocks."""
    results = []
    for block in message.content:
        if block.type != "tool_use":
            continue
        try:
            if block.name not in registry:
                raise ValueError(f"unknown tool {block.name!r}")
            output = registry[block.name](**block.input)
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(output), "is_error": False})
        except Exception as e:
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": f"Error: {e}", "is_error": True})
    return results


def describe_call(block):
    return f"{block.name}({json.dumps(block.input)[:120]})"


def run_conversation(messages, tools, registry, system=None, max_iterations=10):
    for turn in range(1, max_iterations + 1):
        response = chat(messages, system=system, tools=tools)
        add_assistant_message(messages, response)

        calls = [b for b in response.content if b.type == "tool_use"]
        print(f"turn {turn}: stop_reason={response.stop_reason}, {len(calls)} tool call(s)")
        for b in calls:
            print("   ", describe_call(b))

        if response.stop_reason != "tool_use":
            return response
        add_user_message(messages, run_tools(response, registry))   # ONE user message with ALL results
    raise RuntimeError(f"no final answer after {max_iterations} turns")

## 3. Multiple tools and parallel tool calls

With all three tools available, one request that needs two reminders. Watch the trace: when two calls don't depend on each other, Claude asks for **both in the same turn** (parallel tool calls), and `run_tools` answers both in a single user message.

In [4]:
REMINDERS.clear()
messages = []
add_user_message(messages, (
    "Set two reminders for me: renew my passport 6 months from today, "
    "and call the dentist 3 days from today. Both at 09:00."
))
final = run_conversation(messages, reminder_tools, TOOL_FUNCTIONS)

print("\nClaude:", text_of(final))
print("\nStored reminders:", json.dumps(REMINDERS, indent=2))

turn 1: stop_reason=tool_use, 1 tool call(s)
    get_current_datetime({})


turn 2: stop_reason=tool_use, 2 tool call(s)
    add_duration_to_datetime({"datetime_str": "2026-11-15", "duration": 6, "unit": "months"})
    add_duration_to_datetime({"datetime_str": "2026-11-15", "duration": 3, "unit": "days"})


turn 3: stop_reason=tool_use, 2 tool call(s)
    set_reminder({"content": "Renew my passport", "timestamp": "2027-05-15T09:00:00"})
    set_reminder({"content": "Call the dentist", "timestamp": "2026-11-18T09:00:00"})


turn 4: stop_reason=end_turn, 0 tool call(s)

Claude: I've set both reminders:

- **Call the dentist:** Wednesday, November 18, 2026 at 09:00.
- **Renew your passport:** Saturday, May 15, 2027 at 09:00.

Stored reminders: [
  {
    "content": "Renew my passport",
    "timestamp": "2027-05-15T09:00:00"
  },
  {
    "content": "Call the dentist",
    "timestamp": "2026-11-18T09:00:00"
  }
]


Why all results go in **one** user message: the results of a parallel turn belong together, right after the assistant turn that asked for them. Splitting them into several user messages makes the history inconsistent and, as the docs warn, teaches Claude to stop making parallel calls.

## 4. The course's "batch tool", and parallel calls now

When the course was recorded, Claude often asked for one tool per turn even when several were independent, which costs a round trip each. The course's workaround is a **batch tool**: one tool whose input is a *list* of tool calls (name + arguments), so Claude can ask for several at once and your code runs them all.

> **Changed since the course.** Current models make parallel tool calls on their own (the trace above), so the batch tool is rarely needed. The opposite switch exists too: `tool_choice={"type": "auto", "disable_parallel_tool_use": True}` limits Claude to at most one tool call per turn.

The idea, run locally with a hand-written input (no API call):

In [5]:
batch_tool_schema: ToolParam = {
    "name": "batch_tool",
    "description": "Runs several independent tool calls at once. Use it when you need two or more tools whose inputs don't depend on each other.",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string", "description": "Name of the tool to call"},
                        "arguments": {"type": "string", "description": "The tool's arguments, as a JSON string"},
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}


def batch_tool(invocations):
    return [{"tool": i["name"], "output": TOOL_FUNCTIONS[i["name"]](**json.loads(i["arguments"]))} for i in invocations]


print(json.dumps(batch_tool([
    {"name": "add_duration_to_datetime", "arguments": json.dumps({"datetime_str": "2050-01-01", "duration": 6, "unit": "months"})},
    {"name": "add_duration_to_datetime", "arguments": json.dumps({"datetime_str": "2050-01-01", "duration": 3, "unit": "days"})},
]), indent=2))

[
  {
    "tool": "add_duration_to_datetime",
    "output": "Friday, July 01, 2050 12:00:00 AM"
  },
  {
    "tool": "add_duration_to_datetime",
    "output": "Tuesday, January 04, 2050 12:00:00 AM"
  }
]


## 5. Tools for structured data

A tool's input schema is a JSON schema, and Claude fills it in. So a tool you never actually "run" is a way to get structured data: define a tool whose input is the data you want, ask Claude to call it, and read `block.input`. The course **forces** the call with `tool_choice={"type": "tool", "name": ...}`, so Claude must call that tool and can't answer in prose.

> **Changed since the course.** Forced `tool_choice` (`{"type": "any"}` or `{"type": "tool", ...}`) returns a 400 on Opus 5.5 and Sonnet 5.5. (Older models such as Opus 5 and Sonnet 5 still accept it.)

In [6]:
save_meeting_schema: ToolParam = {
    "name": "save_meeting",
    "description": (
        "Saves a meeting extracted from a message to the calendar. Fill every field; "
        "use null for a field the message doesn't state. Don't guess dates or times."
    ),
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "date": {"anyOf": [{"type": "string"}, {"type": "null"}], "description": "YYYY-MM-DD, or null"},
            "time": {"anyOf": [{"type": "string"}, {"type": "null"}], "description": "HH:MM, 24-hour, or null"},
            "location": {"anyOf": [{"type": "string"}, {"type": "null"}]},
            "attendees": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["title", "date", "time", "location", "attendees"],
        "additionalProperties": False,
    },
}

note = "Hi both, quarterly planning lunch with Priya and Marco on 2050-03-04 at Cafe Rio. Bring the Q1 numbers!"

try:
    chat(
        [{"role": "user", "content": f"Extract the meeting from this note:\n\n{note}"}],
        tools=[save_meeting_schema],
        tool_choice={"type": "tool", "name": "save_meeting"},
    )
except anthropic.BadRequestError as e:
    print("API 400:", e.message)

API 400:

 Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'tool_choice: type "tool" and "any" are not supported for this model.'}, 'request_id': 'req_011Ch5VWXrXXwebAXkEqNLYp'}


The current way, when you want the data through a tool:

- leave `tool_choice` on `auto` and **say in the prompt** which tool to use;
- set **`strict: true`** on the tool (schema with `additionalProperties: false` and `required`), so the input is guaranteed to match the schema;
- `auto` doesn't *guarantee* a call, so **check** for a `tool_use` block, and re-prompt if there's none.

When the only goal is JSON (no real tool behind it), structured outputs (`output_config.format`, lesson [2.2](02-streaming-and-structured-data.ipynb)) is simpler.

In [7]:
response = chat(
    [{"role": "user", "content": f"Extract the meeting from this note and save it with the save_meeting tool.\n\n{note}"}],
    tools=[save_meeting_schema],
)
call = next((b for b in response.content if b.type == "tool_use" and b.name == "save_meeting"), None)
if call is None:
    print("No tool call came back; re-prompt. Claude said:", text_of(response))
else:
    print("stop_reason:", response.stop_reason)
    print(json.dumps(call.input, indent=2))

stop_reason: tool_use
{
  "title": "Quarterly planning lunch",
  "date": "2050-03-04",
  "time": null,
  "location": "Cafe Rio",
  "attendees": [
    "Priya",
    "Marco"
  ]
}


The note gives no time, so `time` is null rather than an invented "12:00": the schema allows null, and the description says not to guess.

**Exam angle (Domain 4).** The exam guide (v1.0) still describes forcing a tool as *the* way to guarantee structured output: `tool_choice: "any"` guarantees Claude calls *some* tool (useful when several extraction schemas exist and the document type is unknown); a named tool guarantees *which* one, e.g. to run an extraction step before anything else. Answer exam questions in those terms. In code on current models: `auto` + `strict: true` + an instruction, or structured outputs.

## 6. Fine-grained tool streaming

Out of exam scope (streaming): setting `eager_input_streaming: true` on a tool makes the API stream that tool's input JSON as it is generated, without buffering and validating it first, so you must validate it yourself.

## 7. The text editor tool

Some tools are **defined by Anthropic**: Claude has been trained on them, so you declare them by `type` and `name` only, with no schema. The **text editor tool** lets Claude view and edit files:

```python
{"type": "text_editor_20250728", "name": "str_replace_based_edit_tool"}
```

It is still a **client tool**: Claude sends `tool_use` blocks with a `command` (`view`, `create`, `str_replace`, `insert`) and your code does the file work. This is the mechanism behind coding agents like Claude Code.

The `path` comes from the model, so treat it as **untrusted input**: every path is resolved and must stay inside one sandbox folder (`outputs/06-text-editor-sandbox/`), or the call is rejected.

In [8]:
SANDBOX = (Path("outputs") / "06-text-editor-sandbox").resolve()


def safe_path(path):
    """Resolve a model-supplied path; reject anything outside the sandbox (.., absolute paths, symlinks)."""
    p = Path(path)
    target = (p if p.is_absolute() else SANDBOX / p).resolve()
    if not target.is_relative_to(SANDBOX):
        raise PermissionError(f"{path!r} is outside the sandbox folder")
    return target


def text_editor(command, path, **kwargs):
    target = safe_path(path)
    if command == "view":
        if target.is_dir():
            return "\n".join(sorted(c.name + ("/" if c.is_dir() else "") for c in target.iterdir()))
        lines = target.read_text().splitlines()
        start, end = kwargs.get("view_range") or (1, len(lines))
        end = len(lines) if end == -1 else end
        return "\n".join(f"{i}: {lines[i - 1]}" for i in range(start, end + 1))
    if command == "create":
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(kwargs["file_text"])
        return f"Created {path}"
    if command == "str_replace":
        text = target.read_text()
        count = text.count(kwargs["old_str"])
        if count != 1:
            raise ValueError(f"old_str must match exactly once; it matched {count} times")
        target.write_text(text.replace(kwargs["old_str"], kwargs.get("new_str", ""), 1))
        return "Replaced 1 occurrence"
    if command == "insert":
        lines = target.read_text().splitlines(keepends=True)
        line = kwargs["insert_line"]
        if not 0 <= line <= len(lines):
            raise ValueError(f"insert_line must be between 0 and {len(lines)}")
        new_text = kwargs["insert_text"]
        lines.insert(line, new_text if new_text.endswith("\n") else new_text + "\n")
        target.write_text("".join(lines))
        return f"Inserted text after line {line}"
    raise ValueError(f"unknown command {command!r}")

The guard, tested locally before Claude gets near it:

In [9]:
shutil.rmtree(SANDBOX, ignore_errors=True)
SANDBOX.mkdir(parents=True)
(SANDBOX / "main.py").write_text(
    "def average(numbers):\n"
    "    total = 0\n"
    "    for n in numbers:\n"
    "        total += n\n"
    "    return total / len(numbers)\n"
)

print(text_editor("view", "main.py"))
for bad in ("../../.env", "/etc/passwd", "sub/../../escape.txt"):
    try:
        text_editor("view", bad)
    except PermissionError as e:
        print("PermissionError:", e)

1: def average(numbers):
2:     total = 0
3:     for n in numbers:
4:         total += n
5:     return total / len(numbers)
PermissionError: '../../.env' is outside the sandbox folder
PermissionError: '/etc/passwd' is outside the sandbox folder
PermissionError: 'sub/../../escape.txt' is outside the sandbox folder


Now Claude gets the tool and a small job. The loop and `run_tools` are the same as for the reminders; only the tool list and the registry change.

In [10]:
editor_tools = [{"type": "text_editor_20250728", "name": "str_replace_based_edit_tool"}]
editor_registry = {"str_replace_based_edit_tool": text_editor}

messages = []
add_user_message(messages, (
    "In main.py (use relative paths), average() crashes on an empty list. "
    "Make it return 0.0 for an empty list, and add a one-line docstring. Keep the change minimal."
))
final = run_conversation(messages, editor_tools, editor_registry)

print("\nClaude:", text_of(final))
print("\n--- main.py now ---")
print((SANDBOX / "main.py").read_text())

turn 1: stop_reason=tool_use, 1 tool call(s)
    str_replace_based_edit_tool({"command": "view", "path": "main.py"})


turn 2: stop_reason=tool_use, 1 tool call(s)
    str_replace_based_edit_tool({"command": "str_replace", "path": "main.py", "old_str": "def average(numbers):\n    total = 0", "new_str": "def average)


turn 3: stop_reason=end_turn, 0 tool call(s)

Claude: I changed `average()` in `main.py` so it returns `0.0` for an empty list instead of crashing. I also added a one-line docstring. I haven't run it.

The new code is:

```python
def average(numbers):
    """Return the mean of numbers, or 0.0 if the list is empty."""
    if not numbers:
        return 0.0
    total = 0
    ...
```

The rest of the function is unchanged.

--- main.py now ---
def average(numbers):
    """Return the mean of numbers, or 0.0 if the list is empty."""
    if not numbers:
        return 0.0
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)



## 8. The web search tool

**Server tools** run on Anthropic's side: you add them to `tools`, and the search, the result handling and the answer all happen inside one API call. There's no `tool_use` for you to answer; the response contains `server_tool_use` and `web_search_tool_result` blocks for you to inspect, and text blocks with **citations** to the pages used.

- `max_uses` caps the number of searches in one request (here 2). `allowed_domains` / `blocked_domains` restrict the sources.
- Price: $10 per 1,000 searches, plus the search results as input tokens.
- With the `web_search_20260209` version Claude can also filter results with code before reading them ("dynamic filtering"), so code-execution blocks may appear.
- A long server-side turn can end with `stop_reason == "pause_turn"`: send the conversation back as it is to let it continue.

In [11]:
response = client.messages.create(
    model=model,
    max_tokens=3000,
    tools=[{"type": "web_search_20260209", "name": "web_search", "max_uses": 2}],
    messages=[{"role": "user", "content": "What is the latest stable release of Python? Answer in one or two sentences."}],
)

print("stop_reason:", response.stop_reason)
print("blocks:", [b.type for b in response.content], "\n")
for b in response.content:
    if b.type == "server_tool_use":
        detail = b.input.get("query") or b.input.get("code", "").splitlines()[0]
        print(f"server_tool_use  {b.name:<15} {detail[:90]}")

answer = "".join(b.text for b in response.content if b.type == "text")
print("\nClaude:", answer.strip())

print("\nPages the search returned (first 5):")
for b in response.content:
    if b.type == "web_search_tool_result" and isinstance(b.content, list):
        for result in b.content[:5]:
            print(" -", result.title, "|", result.url)

citations = [c for b in response.content if b.type == "text" for c in (b.citations or [])]
print("\nCitations attached to the answer:", len(citations))
usage = response.usage.server_tool_use
print("Searches billed:", usage.web_search_requests if usage else 0)

stop_reason: end_turn
blocks: ['server_tool_use', 'server_tool_use', 'web_search_tool_result', 'code_execution_tool_result', 'server_tool_use', 'code_execution_tool_result', 'thinking', 'server_tool_use', 'code_execution_tool_result', 'text'] 

server_tool_use  code_execution  r = await web_search({"query": "latest stable Python release python.org downloads"})
server_tool_use  web_search      latest stable Python release python.org downloads
server_tool_use  code_execution  print(type(r), str(r)[:1500])
server_tool_use  code_execution  import json

Claude: The latest stable release I found is **Python 3.15.2**. Both python.org's source releases page and Wikipedia list it as the newest, and Wikipedia gives its release date as 26 October 2026. The search results were somewhat inconsistent about dates, so check python.org/downloads to confirm.

Pages the search returned (first 5):
 - Python Source Releases | https://www.python.org/downloads/source/
 - Python Release Python 3.14.0 | https:

**What this run shows.**

- Claude didn't read raw search results. With the `20260209` version it wrote code (`code_execution`) that ran the search inside Anthropic's sandbox and printed only titles, URLs and a slice of each page, then read that output: the "dynamic filtering" mentioned above. Only one search was billed.
- Because the results reached Claude through code output, this answer came back with **no citation blocks**. If every claim must be cited, check `citations` in code and don't assume they are there.
- The sources disagreed about dates, and Claude **said so** and pointed to the primary source (python.org) instead of silently picking one. That's the behaviour to ask for when synthesising from several sources.

**Exam angle.**

- **Domain 1, the loop:** continue while `stop_reason == "tool_use"`, stop on `"end_turn"`. Don't decide that the agent is done by reading its text ("looks like it finished") or rely on an iteration cap as the main stop; a cap is only a safety net. Every turn appends the full assistant content, and all results of a turn go back in one user message.
- **Domain 2, which tools:** give each agent a small, focused tool set with clear, distinct descriptions; overlapping tools are a common reason for wrong tool choice. Know the three kinds: **your own tools** (you define and run them), **Anthropic-defined client tools** (text editor, bash: built-in schema, you run them, and you own the sandbox and path checks), **server tools** (web search, web fetch, code execution: Anthropic runs them).
- **Domain 5, provenance:** when sources conflict, keep both values with their sources and flag the conflict rather than choosing one silently; keep track of which claim came from which source.
- **Domain 4, structured output via tools:** in exam terms, `tool_choice: "any"` guarantees *a* tool call and a named tool guarantees *that* tool; a JSON schema removes syntax errors but not semantic ones (validate in code).

## Recap

- The agentic loop: call → append full reply → if `tool_use`, run all calls and return all results in one user message → repeat until `end_turn`.
- Parallel tool calls are the default; the course's batch tool is a workaround from before that; `disable_parallel_tool_use` turns them off.
- Forced `tool_choice` returns a 400 on Sonnet 5.5 / Opus 5.5: use `auto` + `strict: true` + an instruction (and check a call came back), or structured outputs.
- The text editor tool is Anthropic-defined but runs on your side: confine paths to a sandbox.
- Web search is a server tool: one request, citations in the text blocks, `max_uses` to cap cost.

Next: [2.7 RAG and agentic search](07-rag-and-agentic-search.ipynb).